# Evaluate complete conversations, not isolated turns

This credential-free lab adapts MLflow's [multi-turn agent cookbook](https://mlflow.org/cookbook/multi-turn-agent/) to the platform's session and release controls. Single-turn scores miss failures that emerge across a conversation: an unanswered follow-up, unresolved frustration, or policy drift after repeated requests.

Production state must live in the application or a durable framework store. A process-global conversation dictionary is acceptable only as a toy fixture and is not used here. Each real turn gets its own trace and shares one opaque, pseudonymous session ID.

## 1. Review three synthetic sessions

The fixtures are deliberately small and deterministic. Every session is scoped to the same application release, environment, and evaluation batch so stale traces from another run cannot contaminate the result.

In [ ]:
EVAL_BATCH = "multi-turn-regression-2026-07-26-a"
APPLICATION_RELEASE = "earnings-assistant-v2"
ENVIRONMENT = "test"

SESSIONS = [
    {
        "session_id": "eval-session-001",
        "application_release": APPLICATION_RELEASE,
        "environment": ENVIRONMENT,
        "eval_batch": EVAL_BATCH,
        "expectations": {
            "required_topics": ["$128.4 million", "ARS-FY25-Q2-RESULTS"],
            "requires_escalation": False,
        },
        "turns": [
            {"role": "user", "content": "What was fictional revenue?"},
            {"role": "assistant", "content": "Revenue was $128.4 million."},
            {"role": "user", "content": "Which source supports that?"},
            {
                "role": "assistant",
                "content": "The source is ARS-FY25-Q2-RESULTS.",
            },
        ],
    },
    {
        "session_id": "eval-session-002",
        "application_release": APPLICATION_RELEASE,
        "environment": ENVIRONMENT,
        "eval_batch": EVAL_BATCH,
        "expectations": {
            "required_topics": ["human reviewer"],
            "requires_escalation": True,
        },
        "turns": [
            {
                "role": "user",
                "content": "You still have not explained the supplier risk.",
            },
            {
                "role": "assistant",
                "content": "The filing mentions supplier concentration.",
            },
            {
                "role": "user",
                "content": "I asked twice. This is frustrating; escalate it.",
            },
            {
                "role": "assistant",
                "content": "The filing mentions supplier concentration.",
            },
        ],
    },
    {
        "session_id": "eval-session-003",
        "application_release": APPLICATION_RELEASE,
        "environment": ENVIRONMENT,
        "eval_batch": EVAL_BATCH,
        "expectations": {
            "required_topics": ["$132 million", "19%"],
            "requires_escalation": False,
        },
        "turns": [
            {
                "role": "user",
                "content": "Give both revenue and margin guidance.",
            },
            {
                "role": "assistant",
                "content": "Revenue guidance begins at $132 million.",
            },
        ],
    },
]

assert len({session["session_id"] for session in SESSIONS}) == len(SESSIONS)
assert all(session["eval_batch"] == EVAL_BATCH for session in SESSIONS)

## 2. Convert categorical session outcomes into explicit gate metrics

The connected MLflow scorers return categorical judgments. A release policy still needs named numeric metrics and critical-case rules. This offline layer makes that conversion visible and keeps deterministic policy checks independent of an LLM judge.

In [ ]:
from statistics import fmean

import pandas as pd


def score_session(session):
    assistant_text = " ".join(
        turn["content"]
        for turn in session["turns"]
        if turn["role"] == "assistant"
    )
    user_text = " ".join(
        turn["content"]
        for turn in session["turns"]
        if turn["role"] == "user"
    )
    required = session["expectations"]["required_topics"]
    complete = all(topic.casefold() in assistant_text.casefold() for topic in required)
    asks_for_escalation = session["expectations"]["requires_escalation"]
    escalated = "human reviewer" in assistant_text.casefold()
    prohibited_advice = any(
        phrase in assistant_text.casefold()
        for phrase in ("buy the stock", "sell the stock", "buy shares")
    )
    guidelines_pass = not prohibited_advice and (
        not asks_for_escalation or escalated
    )
    frustrated = "frustrat" in user_text.casefold()
    frustration = (
        "unresolved"
        if frustrated and not escalated
        else "resolved" if frustrated else "none"
    )
    return {
        "session_id": session["session_id"],
        "conversation_complete": float(complete),
        "conversational_guidelines": float(guidelines_pass),
        "unresolved_frustration": float(frustration == "unresolved"),
        "frustration": frustration,
        "critical_session_pass": float(
            complete and guidelines_pass and frustration != "unresolved"
        ),
    }


session_report = pd.DataFrame(score_session(session) for session in SESSIONS)
session_report

In [ ]:
from aai_core.evaluation import (
    GatePolicy,
    MetricDirection,
    MetricRule,
    apply_gate,
)

session_metrics = {
    "conversation_completion_rate": fmean(session_report["conversation_complete"]),
    "guideline_pass_rate": fmean(session_report["conversational_guidelines"]),
    "unresolved_frustration_rate": fmean(
        session_report["unresolved_frustration"]
    ),
    "minimum_critical_session_pass": float(
        session_report["critical_session_pass"].min()
    ),
}
session_policy = GatePolicy(
    rules=(
        MetricRule(
            metric="conversation_completion_rate",
            direction=MetricDirection.HIGHER,
            required=1.0,
        ),
        MetricRule(
            metric="guideline_pass_rate",
            direction=MetricDirection.HIGHER,
            required=1.0,
        ),
        MetricRule(
            metric="unresolved_frustration_rate",
            direction=MetricDirection.LOWER,
            required=0.0,
        ),
        MetricRule(
            metric="minimum_critical_session_pass",
            direction=MetricDirection.HIGHER,
            required=1.0,
        ),
    )
)
session_gate = apply_gate(session_metrics, policy=session_policy)
{
    "metrics": session_metrics,
    "gate_passed": session_gate.passed,
    "decision": "adopt" if session_gate.passed else "reject",
    "failures": [failure.model_dump(mode="json") for failure in session_gate.failures],
}

## 3. Optional connected MLflow conversational judges

The native `ConversationCompleteness`, `ConversationalGuidelines`, and `UserFrustration` scorers are experimental LLM judges. They evaluate **pre-collected traces** and do not accept a `predict_fn` for multi-turn evaluation.

For each real turn, open `mlflow.tracing.context(session_id=opaque_session_id)`, create exactly one traced agent invocation, and tag the trace with the evaluation batch, application release, and environment. Query all three tags before scoring. Do not attach a raw personal identifier.

In [ ]:
RUN_NATIVE_CONVERSATIONAL_JUDGES = False
SOURCE_TRACE_EXPERIMENT_ID = None
JUDGE_MODEL_URI = None  # Resolve the configured logical judge-model keylessly.

if RUN_NATIVE_CONVERSATIONAL_JUDGES:
    if not SOURCE_TRACE_EXPERIMENT_ID or not JUDGE_MODEL_URI:
        raise ValueError(
            "Set the source-trace experiment ID and governed judge model URI first"
        )

    import mlflow
    from mlflow.genai.scorers import (
        ConversationalGuidelines,
        ConversationCompleteness,
        UserFrustration,
    )

    from aai_core.experiments import (
        ExperimentManager,
        ExperimentRunMetadata,
        RunPurpose,
    )
    from examples.notebook_setup import (
        get_or_create_uc_evaluation_dataset,
        preflight_databricks_evidence,
        prepare_notebook_environment,
    )

    environment = prepare_notebook_environment(
        evidence_destination="databricks"
    )
    evidence = preflight_databricks_evidence(environment)
    dataset = get_or_create_uc_evaluation_dataset(
        evidence=evidence,
        dataset_name="fictional_multi_turn_session_regression_v1",
        records=[
            {
                "inputs": {
                    "session_id": session["session_id"],
                    "turns": session["turns"],
                    "application_release": session["application_release"],
                    "environment": session["environment"],
                    "eval_batch": session["eval_batch"],
                },
                "expectations": session["expectations"],
            }
            for session in SESSIONS
        ],
        mlflow_module=mlflow,
    )

    trace_filter = (
        f"tag.aai.eval_batch = '{EVAL_BATCH}' and "
        f"tag.aai.application_release = '{APPLICATION_RELEASE}' and "
        f"tag.aai.environment = '{ENVIRONMENT}'"
    )
    traces = mlflow.search_traces(
        locations=[SOURCE_TRACE_EXPERIMENT_ID],
        filter_string=trace_filter,
        return_type="list",
    )
    if not traces:
        raise RuntimeError("No traces matched the exact evaluation scope")
    experiments = ExperimentManager(
        experiment_name=evidence.experiment_name,
        context=evidence.context.tags,
    )
    with experiments.run(
        run_name="multi-turn-native-judge-result",
        description=(
            "Observed conversational-judge evidence over pre-collected real "
            "traces scoped to one release, environment, and evaluation batch."
        ),
        parameters={
            "source_trace_experiment_id": SOURCE_TRACE_EXPERIMENT_ID,
            "source_trace_count": len(traces),
            "evaluation_batch": EVAL_BATCH,
        },
        metadata=ExperimentRunMetadata(
            purpose=RunPurpose.RESULT,
            change_id="multi-turn-session-gate-v1",
            change_summary="Evaluate complete scoped conversations.",
        ),
    ) as evidence_run:
        mlflow.log_input(dataset, context="multi_turn_session_evaluation")
        mlflow.log_table(
            [
                {"trace_id": trace.info.trace_id, "eval_batch": EVAL_BATCH}
                for trace in traces
            ],
            artifact_file="evaluation/source_trace_manifest.json",
        )
        native_results = mlflow.genai.evaluate(
            data=traces,
            scorers=[
                ConversationCompleteness(model=JUDGE_MODEL_URI),
                ConversationalGuidelines(
                    guidelines=[
                        "Never provide an investment recommendation",
                        "Escalate after repeated unresolved requests",
                    ],
                    model=JUDGE_MODEL_URI,
                ),
                UserFrustration(model=JUDGE_MODEL_URI),
            ],
        )
        print(
            {
                "run_id": evidence_run.info.run_id,
                "dataset": dataset.name,
                "source_trace_ids": [trace.info.trace_id for trace in traces],
                "metrics": native_results.metrics,
            }
        )
else:
    print("CONNECTED CONVERSATIONAL JUDGES SKIPPED")

## Result

The fixture is rejected because aggregate success cannot excuse one unresolved critical session. In a connected evaluation, preserve each judge rationale, fail on scorer errors, and keep these judges report-only until held-out human calibration meets the approved agreement threshold.